# FinalCheck agent

**Owner:** Edwin (Joao) · **Code:** `src/final_check_agent.py` · **Tests:** `src/test_final_check_agent.py`

Readability rewrites the text *after* the Verifier approved it, so the text a patient actually
reads has not been checked. FinalCheck is the last gate:

```
final text -> check -> pass -------------------------------> done
                    -> fail -> fix -> check again (max 2) -> still failing -> needs_review
```

Each check has two parts, merged into one verdict:

- **Deterministic guards** (no API call): a changed, dropped or invented number, and a negated
  finding stated without its negation.
- **LLM judge:** reads the SOURCE and the final text, using the same instruction as the Verifier.

`faithful` is recomputed after the merge and can only get stricter. If the check itself cannot
run (quota, bad JSON), the text is flagged, never passed.

**What this notebook measures**

1. Guards alone, on all 40 MedAESQA expert answers, no API: false alarms on a faithful text, and
   how many injected number changes they catch.
2. The full agent with a real model, on a small sample: does it pass a faithful text, does it
   catch an injected error, and does the fix remove it?

In [1]:
import json, re, sys, time, hashlib, random
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import final_check_agent as fc
import readability_agent as ra

DATA = ROOT / "data" / "medaesqa_v1.json"
CACHE = ROOT / "data" / "final_check_runs.json"   # data/ is gitignored
FROZEN_SHA = "a062742c1678b4379121ad030a06bda574afd376f331df41575938dd9849c383"
print("repo root:", ROOT.name[:40], "...")

repo root: Google-2D-faithfulmed-a-multi-agent-syst ...


In [2]:
# Same gold set as the baseline notebook. Citation stripping is the baseline's, with one
# fix: PMIDs can be shorter than 6 digits. The baseline's pattern (6 to 9 digits) leaves
# lists like [15857353, 7102534, 37093] in 2 of the 40 answers.
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == FROZEN_SHA, "gold set changed"
items = json.loads(DATA.read_text())

def strip_citations(text):
    """Remove [12345678] markers, then fix the spaces they leave behind."""
    text = re.sub(r"\[\s*\d{1,9}(?:\s*,\s*\d{1,9})*\s*\]", "", text or "")
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

examples = [{"qid": it["question_id"], "question": it["question"],
             "source": strip_citations(it["expert_curated_answer"])} for it in items]
print(len(examples), "examples, frozen hash verified")

40 examples, frozen hash verified


## Part 1. Guards only, no API

Two questions, both answered without spending quota:

- **False alarms.** Give the guards a text identical to its source. Any problem reported is a
  false alarm.
- **Catch rate on number changes.** For every source with a number in it, change one number
  and see if the guards catch it.

The negation guard needs the Extractor's `negated` flags, which this notebook does not have, so
it is tested in `test_final_check_agent.py` and in the pipeline test, not here. Catching a
dropped negation without those flags is the LLM judge's job, measured in Part 2.

In [3]:
NUM = re.compile(r"(?<![\w.])\d+(?:\.\d+)?")

def change_first_number(text, source):
    """Double the first number, making sure the new value is not already in the source."""
    m = next((m for m in NUM.finditer(text) if len(m.group(0)) <= 5), None)  # skip IDs
    if not m:
        return None
    old = m.group(0)
    new = str(float(old) * 2).rstrip("0").rstrip(".") if "." in old else str(int(old) * 2 or 5)
    while new in fc._number_set(source):
        new = str(int(float(new)) + 7)
    return {"text": text[:m.start()] + new + text[m.end():], "detail": f"{old} -> {new}",
            "old": old, "new": new}

false_alarms = [e["qid"] for e in examples
                if fc.problem_count(fc.deterministic_check(e["source"], [], e["source"]))]
with_numbers = [e for e in examples if NUM.search(e["source"])]
caught = [e["qid"] for e in with_numbers
          if fc.problem_count(fc.deterministic_check(
              e["source"], [], change_first_number(e["source"], e["source"])["text"]))]

print(f"false alarms on an unchanged text: {len(false_alarms)} of {len(examples)}")
print(f"number changes caught:            {len(caught)} of {len(with_numbers)} "
      f"sources that contain a number")

false alarms on an unchanged text: 0 of 40
number changes caught:            9 of 9 sources that contain a number


## Part 2. The full agent with a real model

For each example in a small sample:

1. **Readability** rewrites the expert answer (`src/readability_agent.py`). That rewrite is
   exactly what FinalCheck receives in the pipeline.
2. **FinalCheck on the clean rewrite.** It should pass. A failure here is either a real problem
   Readability introduced, or a false alarm. Each one is read by hand below.
3. **FinalCheck on a broken copy.** One error is injected on purpose: a number changed, a
   negation dropped, or a sentence dropped. FinalCheck should catch it, and the fix should
   remove it.

**Quota.** The free tier is 20 requests per day per model. Every call is saved to
`data/final_check_runs.json` as soon as it returns. Re-running this cell skips finished work,
so a quota stop loses nothing. The model each call used is recorded, because a run that falls
back across models has to say so.

In [4]:
def load_key():
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    for p in (ROOT / ".env", Path.cwd() / ".env"):
        if p.exists():
            for line in p.read_text().splitlines():
                if line.startswith("GEMINI_API_KEY="):
                    return line.split("=", 1)[1].strip().strip("'\"")
    import os
    return os.environ.get("GEMINI_API_KEY")

KEY = load_key()
assert KEY and "your" not in KEY.lower(), "Set GEMINI_API_KEY (Colab secret or .env)"
print("API key loaded")

from google import genai
client = genai.Client(api_key=KEY)
# 3.6-flash last: it returned 503 on long prompts on 2026-10-13. A 429 moves straight
# to the next model, so models that are out of quota today cost nothing.
MODELS = ("gemini-3.5-flash", "gemini-3.1-flash-lite", "gemini-3.5-flash-lite",
          "gemini-3.6-flash")
base_fn = fc.make_generate_fn(client, models=MODELS, retries=2, base_delay=5)

CALL_LOG = []
def generate_fn(prompt):
    t = time.time()
    out = base_fn(prompt)
    CALL_LOG.append({"model": out.get("model_used"), "error": out.get("error"),
                     "seconds": round(time.time() - t, 1)})
    return out

API key loaded


In [5]:
NEG = re.compile(r"\b(not|never|no)\s+", re.I)

def drop_negation(text):
    m = NEG.search(text)
    if not m:
        return None
    return {"text": text[:m.start()] + text[m.end():], "detail": f"removed '{m.group(1)}'"}

def drop_sentence(text):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    long_ = [i for i, s in enumerate(sents) if len(s.split()) >= 6]
    if len(sents) < 3 or not long_:
        return None
    i = long_[len(long_) // 2]
    return {"text": " ".join(sents[:i] + sents[i + 1:]), "detail": sents[i][:80]}

def inject(text, source, i):
    """One error per example. The preferred type rotates so all three types get used."""
    kinds = ["number_changed", "negation_dropped", "sentence_dropped"]
    for k in kinds[i % 3:] + kinds[:i % 3]:
        e = (change_first_number(text, source) if k == "number_changed" else
             drop_negation(text) if k == "negation_dropped" else drop_sentence(text))
        if e:
            return {"kind": k, **e}
    return None

def error_removed(kind, err, final, broken):
    """Did the shipped text get rid of the injected error? Checked without the LLM."""
    if final.strip() == broken.strip():
        return False
    if kind == "number_changed":
        return err["new"] not in fc._number_set(final) and err["old"] in fc._number_set(final)
    if kind == "negation_dropped":
        return len(NEG.findall(final)) >= len(NEG.findall(broken)) + 1
    return None   # a restored sentence needs a human read; see the table below

In [6]:
N = 8          # sample size. Sources with numbers or negations first, so every error
               # type has something to inject into. Stated in the write-up.
rank = lambda e: (not NUM.search(e["source"]), not NEG.search(e["source"]), int(e["qid"]))
sample = sorted(examples, key=rank)[:N]

cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
def save():
    CACHE.write_text(json.dumps(cache, indent=1))

def quota_dead(err):
    return bool(err) and ("429" in err or "RESOURCE_EXHAUSTED" in err)

stopped = None
for i, ex in enumerate(sample):
    row = cache.setdefault(ex["qid"], {"qid": ex["qid"]})

    if "readability" not in row:
        r = ra.run(ex["source"], generate_fn)
        if r.error:
            stopped = r.error; break
        row["readability"] = {"text": r.final, "before": r.before, "after": r.after,
                              "passes_used": r.passes_used, "rejected": r.rejected_passes}
        save()
    clean = row["readability"]["text"]

    if "clean" not in row:
        r = fc.run(ex["source"], clean, generate_fn)
        if quota_dead(r.error):          # out of quota mid-run: do not save a partial run
            stopped = r.error; break
        row["clean"] = r.as_dict(); save()

    if "injected" not in row:
        err = inject(clean, ex["source"], i)
        if err is None:
            row["injected"] = None; save(); continue
        r = fc.run(ex["source"], err["text"], generate_fn)
        if quota_dead(r.error):
            stopped = r.error; break
        row["injected"] = {"error": err, "result": r.as_dict(),
                           "removed": error_removed(err["kind"], err, r.final, err["text"])}
        save()

done = [cache[e["qid"]] for e in sample if "injected" in cache.get(e["qid"], {})]
print(f"{len(done)} of {len(sample)} examples finished, {len(CALL_LOG)} calls this session")
if stopped:
    print("stopped early:", stopped[:160])
    print("re-run this cell tomorrow (or with another key) to finish the rest")

8 of 8 examples finished, 31 calls this session


## Results

In [7]:
import pandas as pd

rows = []
for r in done:
    c, inj = r["clean"], r["injected"]
    rows.append({
        "qid": r["qid"],
        "fk_before": r["readability"]["before"]["fk_grade"],
        "fk_after": r["readability"]["after"]["fk_grade"],
        "clean_passed": c["passed"],
        "clean_reason": c["reason"],
        "clean_calls": c["llm_calls"],
        "error_kind": inj["error"]["kind"] if inj else None,
        "error_detail": inj["error"]["detail"] if inj else None,
        "first_check_caught": (inj["result"]["history"][0]["verdict"] or {}).get("faithful") is False
                              if inj else None,
        "inj_passed": inj["result"]["passed"] if inj else None,
        "inj_reason": inj["result"]["reason"] if inj else None,
        "error_removed": inj["removed"] if inj else None,
        "inj_calls": inj["result"]["llm_calls"] if inj else None,
    })
df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 60)
df

,qid,fk_before,fk_after,clean_passed,clean_reason,clean_calls,error_kind,error_detail,first_check_caught,inj_passed,inj_reason,error_removed,inj_calls
0,118,7.58,7.58,True,passed,1,number_changed,24 -> 48,True,True,passed after 1 fix(es),True,3
1,124,12.21,7.67,True,passed,1,negation_dropped,removed 'not',True,True,passed after 1 fix(es),True,3
2,137,13.57,7.80,True,passed,1,sentence_dropped,Researchers also looked at adults with diabetes separately.,False,True,passed,False,1
3,142,14.75,6.81,True,passed,1,number_changed,19 -> 38,True,True,passed after 1 fix(es),True,3
4,152,11.62,7.15,True,passed,1,negation_dropped,removed 'not',True,True,passed after 1 fix(es),True,3
5,153,14.96,7.11,True,passed,1,sentence_dropped,Doctors often used the 10/30 rule (hemoglobin 10 g/dL - ...,True,True,passed after 1 fix(es),None,3
6,132,16.03,6.94,True,passed,1,number_changed,2 -> 4,True,True,passed after 1 fix(es),True,3
7,143,11.58,6.84,True,passed,1,sentence_dropped,"In a small number of people, HD starts late, after 60 ye...",True,True,passed after 1 fix(es),None,3


In [8]:
n = len(df)
inj = df[df.error_kind.notna()]
models = {}
for r in done:
    for part in (r["clean"], (r["injected"] or {}).get("result") or {}):
        for m in part.get("models_used", []):
            models[m] = models.get(m, 0) + 1

print(f"examples finished: {n}")
print(f"clean rewrite passed:         {int(df.clean_passed.sum())} of {n}")
print(f"injected error caught at once: {int(inj.first_check_caught.sum())} of {len(inj)}")
for k, g in inj.groupby("error_kind"):
    print(f"   {k:17s} caught {int(g.first_check_caught.sum())} of {len(g)}")
print(f"injected error removed by the fix (checked without the LLM): "
      f"{int((inj.error_removed == True).sum())} of {int(inj.error_removed.notna().sum())} "
      f"checkable; sentence drops need a human read")
print(f"flagged needs_review after an injected error: {int((~inj.inj_passed.astype(bool)).sum())} "
      f"of {len(inj)}")
print(f"FinalCheck calls per clean text: mean {df.clean_calls.mean():.1f}; "
      f"per broken text: mean {inj.inj_calls.mean():.1f} (cap is 5)")
print("FinalCheck calls by model:", models)
verdicts = [h["verdict"] for r in done
            for part in (r["clean"], (r["injected"] or {}).get("result") or {"history": []})
            for h in part["history"] if h["verdict"]]
print(f"model said 'not faithful' but named no problem: "
      f"{sum(v.get('unexplained_fail', False) for v in verdicts)} of {len(verdicts)} checks")

examples finished: 8
clean rewrite passed:         8 of 8
injected error caught at once: 7 of 8
   negation_dropped  caught 2 of 2
   number_changed    caught 3 of 3
   sentence_dropped  caught 2 of 3
injected error removed by the fix (checked without the LLM): 5 of 6 checkable; sentence drops need a human read
flagged needs_review after an injected error: 0 of 8
FinalCheck calls per clean text: mean 1.0; per broken text: mean 2.8 (cap is 5)
FinalCheck calls by model: {'gemini-3.1-flash-lite': 18, 'gemini-3.5-flash-lite': 12}
model said 'not faithful' but named no problem: 4 of 23 checks


In [9]:
# Read every failure by hand. A clean rewrite that failed is either a real problem
# Readability introduced, or a false alarm, and the number alone cannot tell which.
for r in done:
    for label, res in (("clean rewrite", r["clean"]),
                       ("injected error", (r["injected"] or {}).get("result"))):
        if not res or res["passed"]:
            continue
        print(f"--- qid {r['qid']}, {label}: {res['reason']}")
        for h in res["history"]:
            v = h["verdict"] or {}
            for k in ("unsupported_claims", "omissions", "negation_errors"):
                for x in v.get(k, []):
                    print(f"   round {h['round']} {k}: {x[:150]}")
            if h["error"]:
                print(f"   round {h['round']} error: {h['error'][:150]}")

## Findings, run of 2026-10-26

- **8 of 8 examples finished.** 31 calls, on gemini-3.1-flash-lite (18 FinalCheck calls) and
  gemini-3.5-flash-lite (12). gemini-3.5-flash was out of daily quota.
- **Faithful rewrites: 8 of 8 passed on the first check**, one call each. No false alarms.
- **Injected errors caught on the first check: 7 of 8.** Numbers 3 of 3, negations 2 of 2,
  dropped sentences 2 of 3.
- **The one miss:** qid 137, dropped sentence "Researchers also looked at adults with
  diabetes separately." It has no number and no negation, so only the LLM judge could catch
  it, and it did not.
- **Every caught error was fixed in one round.** 5 of 5 confirmed by the script check above.
  The 2 dropped sentences (qid 153 and 143) were restored, confirmed by reading the fixed text.
- **No example ended in `needs_review`.**
- 4 of 23 checks said "not faithful" without naming a problem. These pass, because there is
  nothing to fix (see `decisions` in the module docstring, rule 2).

### What the user sees

Per the v2 diagram, the output is the explanation **plus a faithfulness report**, flagged if it
still fails. Two examples: one clean rewrite and one with an injected error.

In [10]:
from final_check_agent import FinalCheckResult

def show(label, res_dict):
    r = FinalCheckResult(**{k: v for k, v in res_dict.items()
                            if k in FinalCheckResult.__dataclass_fields__})
    rep = r.report()
    print(f"=== {label}: {rep['status'].upper()}")
    print(rep["summary"])
    for x in rep["fixed"]:
        print("  fixed:", x[:120])
    for x in rep["remaining"]:
        print("  still there:", x[:120])
    if rep["terms_not_explained"]:
        print("  terms without a plain explanation:", ", ".join(rep["terms_not_explained"][:6]))
    print(f"  {rep['reading_level']}")
    print()

if done:
    show(f"qid {done[0]['qid']}, clean rewrite", done[0]["clean"])
    first_inj = next((r for r in done if r["injected"]), None)
    if first_inj:
        show(f"qid {first_inj['qid']}, {first_inj['injected']['error']['kind']}",
             first_inj["injected"]["result"])

=== qid 118, clean rewrite: PASSED
Checked against the original text. Nothing was added, changed or left out.
  terms without a plain explanation: corneal
  School grade 7.6 (a typical grade 8 student can read it). Goal: grade 8 or lower, meets the goal.

=== qid 118, number_changed: FIXED
Checked against the original text. Problems were found, the text was corrected (1 time(s)), and the corrected text passed.
  fixed: Most corneal injuries heal in 48 to 72 hours.
  fixed: number 48 is not in the source
  fixed: Most corneal injuries heal in 24 to 72 hours.
  terms without a plain explanation: corneal
  School grade 7.6 (a typical grade 8 student can read it). Goal: grade 8 or lower, meets the goal.



## Limitations

- **Small n.** The free tier allows 20 calls per model per day, and one example costs up to 12
  calls. These numbers show the agent works end to end on real data. They are not yet a
  precision or recall estimate.
- **Mixed models.** When a model runs out of quota the call falls back to the next one. The
  calls-by-model line above says exactly which model produced which numbers.
- **Injected errors are simple.** A changed number, a dropped "not", a dropped sentence.
  Real Readability errors can be subtler, such as a softened warning or a changed time frame.
- **No Extractor facts here.** The negation guard needs the Extractor's `negated` flags, so in
  this notebook dropped negations are caught by the LLM judge alone. In the pipeline, both
  run.